# 00 — Data Exploration
## MedIQ — Hospital Patient Analytics & Readmission Predictor
**Goal:** Understand the diabetic patient dataset structure, identify key features, missing values, and plan the full analytics pipeline.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

os.makedirs('../screenshots', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../exports', exist_ok=True)

# Load raw data
df = pd.read_csv('../data/raw/diabetic_data.csv')

print(f"✅ Data loaded")
print(f"   Shape   : {df.shape}")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print(f"\nColumns: {df.columns.tolist()}")

✅ Data loaded
   Shape   : (101766, 50)
   Rows    : 101,766
   Columns : 50

Columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [2]:
print("=== DATA TYPES ===")
print(df.dtypes.value_counts())

print("\n=== TARGET VARIABLE ===")
print(df['readmitted'].value_counts())
print(f"\nReadmission rate: {(df['readmitted'] != 'NO').mean()*100:.1f}%")

print("\n=== MISSING VALUES (shown as ?) ===")
# This dataset uses '?' for missing values
for col in df.columns:
    missing = (df[col] == '?').sum()
    if missing > 0:
        pct = missing/len(df)*100
        print(f"   {col:<30} {missing:>6,} missing ({pct:.1f}%)")

print("\n=== SAMPLE DATA ===")
df.head(3)

=== DATA TYPES ===
str      37
int64    13
Name: count, dtype: int64

=== TARGET VARIABLE ===
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Readmission rate: 46.1%

=== MISSING VALUES (shown as ?) ===
   race                            2,273 missing (2.2%)
   weight                         98,569 missing (96.9%)
   payer_code                     40,256 missing (39.6%)
   medical_specialty              49,949 missing (49.1%)
   diag_1                             21 missing (0.0%)
   diag_2                            358 missing (0.4%)
   diag_3                          1,423 missing (1.4%)

=== SAMPLE DATA ===


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


In [3]:
# Drop columns with too many missing values or not useful
drop_cols = [
    'weight',           # 96.9% missing
    'payer_code',       # 39.6% missing, not clinical
    'encounter_id',     # just ID
    'patient_nbr',      # just ID
    'examide',          # all same value
    'citoglipton',      # all same value
    'glimepiride-pioglitazone',   # rare drug combo
    'metformin-rosiglitazone',    # rare drug combo
    'metformin-pioglitazone',     # rare drug combo
    'glipizide-metformin',        # rare drug combo
    'glyburide-metformin',        # rare drug combo
]
df = df.drop(columns=drop_cols)

# Replace '?' with NaN
df = df.replace('?', np.nan)

# Drop rows where diag_1 is missing (only 21 rows)
df = df.dropna(subset=['diag_1'])

# Fill medical_specialty and race missing with 'Unknown'
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
df['race'] = df['race'].fillna('Unknown')

# Convert target to binary — <30 days = 1 (high risk), else = 0
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

print(f"✅ Basic cleaning done")
print(f"   Shape after cleaning : {df.shape}")
print(f"   Columns remaining    : {df.shape[1]}")
print(f"\nTarget distribution:")
print(df['readmitted_binary'].value_counts())
print(f"\nEarly readmission rate: {df['readmitted_binary'].mean()*100:.1f}%")

✅ Basic cleaning done
   Shape after cleaning : (101745, 40)
   Columns remaining    : 40

Target distribution:
readmitted_binary
0    90393
1    11352
Name: count, dtype: int64

Early readmission rate: 11.2%


In [4]:
# Age — convert range to midpoint number
age_map = {
    '[0-10)':5, '[10-20)':15, '[20-30)':25, '[30-40)':35,
    '[40-50)':45, '[50-60)':55, '[60-70)':65, '[70-80)':75,
    '[80-90)':85, '[90-100)':95
}
df['age_numeric'] = df['age'].map(age_map)

# Diagnosis — simplify diag_1 to disease category
def categorize_diag(diag):
    try:
        code = float(str(diag).replace('E','').replace('V',''))
        if 390 <= code <= 459 or code == 785: return 'Circulatory'
        elif 460 <= code <= 519 or code == 786: return 'Respiratory'
        elif 520 <= code <= 579 or code == 787: return 'Digestive'
        elif code == 250: return 'Diabetes'
        elif 800 <= code <= 999: return 'Injury'
        elif 710 <= code <= 739: return 'Musculoskeletal'
        elif 580 <= code <= 629 or code == 788: return 'Genitourinary'
        elif 140 <= code <= 239: return 'Neoplasms'
        else: return 'Other'
    except:
        return 'Other'

df['diag_1_category'] = df['diag_1'].apply(categorize_diag)

# Medication change features
med_cols = ['metformin','repaglinide','nateglinide','chlorpropamide',
            'glimepiride','glipizide','glyburide','pioglitazone',
            'rosiglitazone','acarbose','insulin']

# Count number of medications changed
df['num_meds_changed'] = df[med_cols].isin(['Up','Down']).sum(axis=1)

# Count number of medications prescribed (not 'No')
df['num_meds_active'] = df[med_cols].isin(['Up','Down','Steady']).sum(axis=1)

# Hospital utilization score
df['utilization_score'] = (
    df['number_outpatient'] +
    df['number_emergency'] * 2 +
    df['number_inpatient'] * 3
)

print(f"✅ Feature engineering complete")
print(f"\nNew features added:")
print(f"   age_numeric       : {df['age_numeric'].describe().to_dict()}")
print(f"\nDiagnosis categories:")
print(df['diag_1_category'].value_counts())
print(f"\nMeds changed range: {df['num_meds_changed'].min()} - {df['num_meds_changed'].max()}")
print(f"Utilization score range: {df['utilization_score'].min()} - {df['utilization_score'].max()}")

✅ Feature engineering complete

New features added:
   age_numeric       : {'count': 101745.0, 'mean': 65.96682883679787, 'std': 15.940959167760255, 'min': 5.0, '25%': 55.0, '50%': 65.0, '75%': 75.0, 'max': 95.0}

Diagnosis categories:
diag_1_category
Circulatory        30437
Other              26693
Respiratory        14423
Digestive           9475
Injury              6975
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Diabetes             235
Name: count, dtype: int64

Meds changed range: 0 - 4
Utilization score range: 0 - 160


In [5]:
# Save clean processed dataset
df.to_csv('../data/processed/mediq_clean.csv', index=False)

print(f"✅ Processed data saved")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {len(df.columns)}")
print(f"\nFinal columns:")
for col in df.columns:
    print(f"   {col}")

✅ Processed data saved
   Rows    : 101,745
   Columns : 45

Final columns:
   race
   gender
   age
   admission_type_id
   discharge_disposition_id
   admission_source_id
   time_in_hospital
   medical_specialty
   num_lab_procedures
   num_procedures
   num_medications
   number_outpatient
   number_emergency
   number_inpatient
   diag_1
   diag_2
   diag_3
   number_diagnoses
   max_glu_serum
   A1Cresult
   metformin
   repaglinide
   nateglinide
   chlorpropamide
   glimepiride
   acetohexamide
   glipizide
   glyburide
   tolbutamide
   pioglitazone
   rosiglitazone
   acarbose
   miglitol
   troglitazone
   tolazamide
   insulin
   change
   diabetesMed
   readmitted
   readmitted_binary
   age_numeric
   diag_1_category
   num_meds_changed
   num_meds_active
   utilization_score
